In [1]:
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

c:\Users\use\Desktop\Credit Card Fraud Detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path.cwd().parent

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"
MODEL_COMPARISON_PATH = REPORTS_DIR / "model_comparison.csv"
MLFLOW_REPORT_PATH = REPORTS_DIR / "mlflow_tracking_report.md"

MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"
MLFLOW_EXPERIMENT_NAME = "fraud-detection-anomaly-analysis"

VALIDATION_METRIC_COLUMNS = [
    "precision",
    "recall",
    "f1_score",
    "pr_auc",
    "roc_auc",
    "false_positive_rate",
]

PARAMETER_COLUMNS = [
    "model",
    "feature_configuration",
    "imbalance_strategy",
    "preprocessing",
]

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("MLflow tracking URI: %s", MLFLOW_TRACKING_URI)
logger.info("MLflow experiment: %s", MLFLOW_EXPERIMENT_NAME)

2026-11-01 12:37:45,795 | INFO | Project root: c:\Users\use\Desktop\Credit Card Fraud Detection
2026-11-01 12:37:45,797 | INFO | MLflow tracking URI: sqlite:///c:/Users/use/Desktop/Credit Card Fraud Detection/mlflow.db
2026-11-01 12:37:45,800 | INFO | MLflow experiment: fraud-detection-anomaly-analysis


In [3]:
def load_and_validate_model_comparison(
    comparison_path: Path,
) -> pd.DataFrame:
    """Load and validate the consolidated experiment results."""
    if not comparison_path.exists():
        raise FileNotFoundError(
            f"Model comparison report not found: {comparison_path}"
        )

    results = pd.read_csv(comparison_path)

    required_columns = {
        "experiment_name",
        "model",
        "feature_configuration",
        "precision",
        "recall",
        "f1_score",
        "pr_auc",
        "roc_auc",
        "false_positive_rate",
    }

    missing_columns = required_columns - set(results.columns)
    if missing_columns:
        raise ValueError(
            f"Model comparison report is missing columns: {sorted(missing_columns)}"
        )

    valid_results = results[
        results["experiment_name"].notna()
        & results["experiment_name"].astype(str).str.strip().ne("")
    ].copy()

    valid_results["experiment_name"] = valid_results[
        "experiment_name"
    ].astype(str).str.strip()

    if valid_results.empty:
        raise ValueError("No valid experiment rows were found.")

    return valid_results


experiment_results = load_and_validate_model_comparison(
    comparison_path=MODEL_COMPARISON_PATH,
)

logger.info("Loaded %s experiment results.", len(experiment_results))
logger.info(
    "Available columns: %s",
    list(experiment_results.columns),
)

display_columns = [
    column
    for column in [
        "experiment_name",
        "model",
        "feature_configuration",
        "imbalance_strategy",
        "precision",
        "recall",
        "f1_score",
        "pr_auc",
        "roc_auc",
    ]
    if column in experiment_results.columns
]

experiment_results[display_columns]

2026-11-01 12:37:45,862 | INFO | Loaded 7 experiment results.
2026-11-01 12:37:45,864 | INFO | Available columns: ['precision', 'recall', 'f1_score', 'pr_auc', 'roc_auc', 'false_positive_rate', 'true_positives', 'false_positives', 'true_negatives', 'false_negatives', 'transactions_flagged', 'experiment_name', 'model', 'feature_configuration', 'preprocessing', 'score_direction', 'training_duration_seconds', 'feature_count']


,experiment_name,model,feature_configuration,precision,recall,f1_score,pr_auc,roc_auc
6,Logistic Regression | Class weighting,Logistic Regression,log_amount_and_time_hours,0.050602,0.887324,0.095745,0.706577,0.968105
7,Random Forest | Class weighting,Random Forest,log_amount_and_time_hours,0.963636,0.746479,0.841270,0.849424,0.945599
8,LightGBM | Positive-class weighting,LightGBM,log_amount_and_time_hours,0.076159,0.647887,0.136296,0.049930,0.817361
9,Logistic Regression | SMOTE,Logistic Regression,log_amount_and_time_hours,0.047800,0.887324,0.090713,0.709086,0.969187
10,Random Forest | SMOTE,Random Forest,log_amount_and_time_hours,0.888889,0.788732,0.835821,0.840678,0.961678
11,Isolation Forest | Baseline,Isolation Forest,log_amount_and_time_hours,0.222222,0.112676,0.149533,0.079585,0.933193
12,Local Outlier Factor | Baseline,Local Outlier Factor,log_amount_and_time_hours,0.000000,0.000000,0.000000,0.002650,0.514945


In [4]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)

if experiment is None:
    raise RuntimeError("MLflow experiment was not created successfully.")

logger.info(
    "MLflow experiment configured: %s (ID: %s)",
    experiment.name,
    experiment.experiment_id,
)

2026-11-01 12:37:48,135 | INFO | MLflow experiment configured: fraud-detection-anomaly-analysis (ID: 1)


In [5]:
import tempfile

def resolve_experiment_artifacts(
    experiment_name: str,
) -> list[Path]:
    """Resolve available metadata and report artifacts for an experiment."""
    artifact_candidates = [
        REPORTS_DIR / "class_imbalance_report.md",
        REPORTS_DIR / "anomaly_detection_report.md",
        REPORTS_DIR / "supervised_candidate_selection_report.md",
        REPORTS_DIR / "supervised_vs_anomaly_comparison.csv",
        REPORTS_DIR / "model_agreement_analysis.csv",
        REPORTS_DIR / "shap_explainability_report.md",
    ]

    return [path for path in artifact_candidates if path.exists()]


def log_experiment_to_mlflow(
    experiment_row: pd.Series,
    experiment_id: str,
) -> str:
    """Log one completed experiment as an MLflow run."""
    experiment_name = str(experiment_row["experiment_name"])

    safe_run_name = (
        experiment_name.lower()
        .replace(" | ", "_")
        .replace(" ", "_")
        .replace("-", "_")
    )

    with mlflow.start_run(
        run_name=safe_run_name,
        experiment_id=experiment_id,
        tags={
            "project": "credit_card_fraud_detection",
            "evaluation_set": "validation",
            "model_selection_status": "validation_only",
            "test_set_used": "false",
        },
    ) as run:
        for parameter_column in PARAMETER_COLUMNS:
            if parameter_column in experiment_row and pd.notna(
                experiment_row[parameter_column]
            ):
                mlflow.log_param(
                    parameter_column,
                    str(experiment_row[parameter_column]),
                )

        mlflow.log_param("experiment_name", experiment_name)
        mlflow.log_param("decision_threshold", 0.5)
        mlflow.log_param("random_seed", 42)

        for metric_column in VALIDATION_METRIC_COLUMNS:
            if metric_column in experiment_row and pd.notna(
                experiment_row[metric_column]
            ):
                mlflow.log_metric(
                    metric_column,
                    float(experiment_row[metric_column]),
                )

        for confusion_column in [
            "true_positives",
            "false_positives",
            "true_negatives",
            "false_negatives",
            "transactions_flagged",
        ]:
            if confusion_column in experiment_row and pd.notna(
                experiment_row[confusion_column]
            ):
                mlflow.log_metric(
                    confusion_column,
                    float(experiment_row[confusion_column]),
                )

        artifact_paths = resolve_experiment_artifacts(experiment_name)

        for artifact_path in artifact_paths:
            mlflow.log_artifact(str(artifact_path), artifact_path="reports")

        experiment_record = experiment_row.where(
            pd.notna(experiment_row),
            None,
        ).to_dict()

        with tempfile.TemporaryDirectory() as temporary_directory:
            experiment_record_path = (
                Path(temporary_directory) / "experiment_record.json"
            )

            with experiment_record_path.open("w", encoding="utf-8") as file:
                json.dump(
                    experiment_record,
                    file,
                    indent=2,
                    default=str,
                )

            mlflow.log_artifact(
                str(experiment_record_path),
                artifact_path="metadata",
            )

        return run.info.run_id


logged_run_ids = {}

for _, experiment_row in experiment_results.iterrows():
    experiment_name = str(experiment_row["experiment_name"])
    logger.info("Logging experiment to MLflow: %s", experiment_name)

    run_id = log_experiment_to_mlflow(
        experiment_row=experiment_row,
        experiment_id=experiment.experiment_id,
    )

    logged_run_ids[experiment_name] = run_id

logger.info("Logged %s experiments to MLflow.", len(logged_run_ids))

2026-11-01 12:37:48,178 | INFO | Logging experiment to MLflow: Logistic Regression | Class weighting
2026-11-01 12:37:49,294 | INFO | Logging experiment to MLflow: Random Forest | Class weighting
2026-11-01 12:37:49,817 | INFO | Logging experiment to MLflow: LightGBM | Positive-class weighting
2026-11-01 12:37:50,277 | INFO | Logging experiment to MLflow: Logistic Regression | SMOTE
2026-11-01 12:37:50,780 | INFO | Logging experiment to MLflow: Random Forest | SMOTE
2026-11-01 12:37:51,245 | INFO | Logging experiment to MLflow: Isolation Forest | Baseline
2026-11-01 12:37:51,720 | INFO | Logging experiment to MLflow: Local Outlier Factor | Baseline
2026-11-01 12:37:52,188 | INFO | Logged 7 experiments to MLflow.


In [6]:
SELECTED_MODEL_NAME = "Random Forest | Class weighting"
SELECTED_MODEL_METADATA_PATH = (
    PROJECT_ROOT
    / "models"
    / "supervised"
    / "class_imbalance_models"
    / "random_forest_class_weighting"
    / "metadata.json"
)
SELECTED_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "supervised"
    / "class_imbalance_models"
    / "random_forest_class_weighting"
    / "model.joblib"
)

if not SELECTED_MODEL_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Selected model metadata not found: {SELECTED_MODEL_METADATA_PATH}"
    )

if not SELECTED_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Selected model artifact not found: {SELECTED_MODEL_PATH}"
    )

with SELECTED_MODEL_METADATA_PATH.open("r", encoding="utf-8") as file:
    selected_model_metadata = json.load(file)

with mlflow.start_run(
    run_name="selected_supervised_candidate",
    experiment_id=experiment.experiment_id,
    tags={
        "project": "credit_card_fraud_detection",
        "model_role": "selected_supervised_candidate",
        "evaluation_set": "validation",
        "test_set_used": "false",
    },
) as selected_model_run:
    mlflow.log_params(
        {
            "selected_model": SELECTED_MODEL_NAME,
            "feature_configuration": selected_model_metadata.get(
                "feature_configuration",
                "unknown",
            ),
            "imbalance_strategy": selected_model_metadata.get(
                "imbalance_strategy",
                "unknown",
            ),
            "decision_threshold": selected_model_metadata.get(
                "decision_threshold",
                0.5,
            ),
            "random_seed": selected_model_metadata.get(
                "random_seed",
                42,
            ),
        }
    )

    validation_metrics = selected_model_metadata.get("validation_metrics", {})

    for metric_name in VALIDATION_METRIC_COLUMNS:
        if metric_name in validation_metrics:
            mlflow.log_metric(
                metric_name,
                float(validation_metrics[metric_name]),
            )

    mlflow.log_artifact(str(SELECTED_MODEL_METADATA_PATH), artifact_path="model_metadata")
    mlflow.log_artifact(str(SELECTED_MODEL_PATH), artifact_path="model")

    selected_model_run_id = selected_model_run.info.run_id

logger.info(
    "Selected supervised candidate logged to MLflow. Run ID: %s",
    selected_model_run_id,
)

2026-11-01 12:37:52,499 | INFO | Selected supervised candidate logged to MLflow. Run ID: 36723fd06c664c01af482203907f7f95


In [7]:
experiment_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
)

logger.info("Found %s MLflow runs.", len(experiment_runs))

experiment_runs[
    [
        "tags.mlflow.runName",
        "params.experiment_name",
        "params.model",
        "params.imbalance_strategy",
        "metrics.pr_auc",
        "metrics.recall",
        "metrics.precision",
        "metrics.false_positives",
        "metrics.false_negatives",
    ]
]

2026-11-01 12:37:52,597 | INFO | Found 9 MLflow runs.


,tags.mlflow.runName,params.experiment_name,params.model,params.imbalance_strategy,metrics.pr_auc,metrics.recall,metrics.precision,metrics.false_positives,metrics.false_negatives
0,selected_supervised_candidate,None,None,Class weighting,0.849424,0.746479,0.963636,NaN,NaN
1,local_outlier_factor_baseline,Local Outlier Factor | Baseline,Local Outlier Factor,None,0.002650,0.000000,0.000000,47.0,71.0
2,isolation_forest_baseline,Isolation Forest | Baseline,Isolation Forest,None,0.079585,0.112676,0.222222,28.0,63.0
3,random_forest_smote,Random Forest | SMOTE,Random Forest,None,0.840678,0.788732,0.888889,7.0,15.0
4,logistic_regression_smote,Logistic Regression | SMOTE,Logistic Regression,None,0.709086,0.887324,0.047800,1255.0,8.0
5,lightgbm_positive_class_weighting,LightGBM | Positive-class weighting,LightGBM,None,0.049930,0.647887,0.076159,558.0,25.0
6,random_forest_class_weighting,Random Forest | Class weighting,Random Forest,None,0.849424,0.746479,0.963636,2.0,18.0
7,logistic_regression_class_weighting,Logistic Regression | Class weighting,Logistic Regression,None,0.706577,0.887324,0.050602,1182.0,8.0
8,logistic_regression_class_weighting,Logistic Regression | Class weighting,Logistic Regression,None,0.706577,0.887324,0.050602,1182.0,8.0


In [8]:
def write_mlflow_tracking_report(
    experiment_results: pd.DataFrame,
    logged_run_ids: dict[str, str],
    selected_model_run_id: str,
    report_path: Path,
) -> None:
    """Write a concise MLflow tracking report."""
    report_lines = [
        "# MLflow Experiment Tracking Report",
        "",
        "## Tracking Configuration",
        "",
        f"- Tracking URI: `{MLFLOW_TRACKING_URI}`",
        f"- Experiment name: `{MLFLOW_EXPERIMENT_NAME}`",
        f"- Experiment ID: `{experiment.experiment_id}`",
        f"- Tracking report generated: {datetime.now(timezone.utc).isoformat()}",
        "- Evaluation dataset: Validation set",
        "- Test set: Not used",
        "",
        "## Logged Experiments",
        "",
    ]

    for _, row in experiment_results.iterrows():
        experiment_name = str(row["experiment_name"])
        run_id = logged_run_ids.get(experiment_name, "Not logged")

        report_lines.extend(
            [
                f"### {experiment_name}",
                "",
                f"- MLflow run ID: `{run_id}`",
                f"- Model: {row.get('model', 'Unknown')}",
                f"- Feature configuration: {row.get('feature_configuration', 'Unknown')}",
                f"- Imbalance strategy: {row.get('imbalance_strategy', 'None')}",
                f"- Validation PR-AUC: {row['pr_auc']:.4f}",
                f"- Validation recall: {row['recall']:.4f}",
                f"- Validation precision: {row['precision']:.4f}",
                "",
            ]
        )

    report_lines.extend(
        [
            "## Selected Supervised Candidate",
            "",
            f"- Model: `{SELECTED_MODEL_NAME}`",
            f"- MLflow run ID: `{selected_model_run_id}`",
            "- Status: Validation candidate; not yet the final production model",
            "",
            "## Important Notes",
            "",
            "- All logged metrics are validation metrics.",
            "- The test set was not used for model selection or threshold optimization.",
            "- The selected model artifact is logged for reproducibility and inspection.",
            "- Final test evaluation must occur only after threshold selection is complete.",
        ]
    )

    report_path.write_text("\n".join(report_lines), encoding="utf-8")


write_mlflow_tracking_report(
    experiment_results=experiment_results,
    logged_run_ids=logged_run_ids,
    selected_model_run_id=selected_model_run_id,
    report_path=MLFLOW_REPORT_PATH,
)

logger.info(
    "MLflow tracking report saved to: %s",
    MLFLOW_REPORT_PATH,
)

2026-11-01 12:37:52,684 | INFO | MLflow tracking report saved to: c:\Users\use\Desktop\Credit Card Fraud Detection\reports\mlflow_tracking_report.md


In [9]:
required_mlflow_artifacts = [
    MODEL_COMPARISON_PATH,
    MLFLOW_REPORT_PATH,
    SELECTED_MODEL_METADATA_PATH,
    SELECTED_MODEL_PATH,
]

missing_artifacts = [
    path for path in required_mlflow_artifacts if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        f"Missing MLflow tracking artifacts: {[str(path) for path in missing_artifacts]}"
    )

if len(logged_run_ids) != len(experiment_results):
    raise RuntimeError(
        f"Expected to log {len(experiment_results)} experiments, "
        f"but logged {len(logged_run_ids)}."
    )

if not experiment_runs.empty:
    logger.info(
        "MLflow history contains %s runs.",
        len(experiment_runs),
    )

logger.info("All required MLflow tracking artifacts were generated successfully.")
print("Phase 7 MLflow experiment tracking completed successfully.")

2026-11-01 12:37:52,720 | INFO | MLflow history contains 9 runs.
2026-11-01 12:37:52,722 | INFO | All required MLflow tracking artifacts were generated successfully.


Phase 7 MLflow experiment tracking completed successfully.
